# Forecasting SPY volatility: naive vs HAR vs GARCH vs XGBoost

**Question:** Can a machine-learning model forecast next week's SPY volatility better than the classic econometric models, out of sample?

**Setup**
- **Target:** realized variance over the next 5 trading days, $RV_{t+1:t+5} = \sum_{i=1}^{5} r_{t+i}^2$ (daily log returns, in %).
- **Models:** each forecasts that target using only information available at the close of day $t$.
  1. **Naive:** the last 22 days' average squared return × 5.
  2. **HAR-RV** (Corsi, 2009): a linear regression on daily, weekly and monthly realized variance. This is the standard benchmark in the volatility literature, and the one ML actually needs to beat.
  3. **GJR-GARCH(1,1)** with Student-t errors. The GJR term lets negative returns raise volatility more than positive ones (the "leverage effect").
  4. **XGBoost** on lagged volatility features, past returns and the VIX.
- **Out-of-sample protocol:** walk-forward. Every model is refit on an expanding window using only past data, and forecasts start in `TEST_START`. The test period is never used for fitting or tuning.
- **Scoring:** MSE and **QLIKE**. QLIKE is the preferred loss for volatility because it stays reliable when the "true" variance is measured with noise (Patton, 2011). A Diebold-Mariano test checks whether the differences are statistically significant.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from arch import arch_model
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 4.5)

# ---- Configuration ----
TICKER = "SPY"
VIX_TICKER = "^VIX"
START = "2000-01-01"
TEST_START = "2015-01-01"   # forecasts are scored from here on
HORIZON = 5                 # forecast the next 5 trading days
GARCH_REFIT_EVERY = 21      # trading days between GARCH refits
ML_REFIT_EVERY = 63         # trading days between HAR / XGBoost refits
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 1. Data
Prices are downloaded once with `yfinance` and cached to `data/` as parquet files, so reruns are instant and use exactly the same data. Delete the folder to refresh.

In [ ]:
def load_close(ticker: str) -> pd.Series:
    path = DATA_DIR / f"{ticker.replace('^', '')}.parquet"
    if path.exists():
        return pd.read_parquet(path)["close"]
    import yfinance as yf
    df = yf.download(ticker, start=START, auto_adjust=True, progress=False)
    if df.empty:
        raise RuntimeError(f"No data downloaded for {ticker}; check your connection.")
    close = df["Close"].squeeze().rename("close")   # newer yfinance returns multi-level columns
    close.index = pd.to_datetime(close.index).tz_localize(None)
    close.to_frame().to_parquet(path)
    return close

spy = load_close(TICKER)
vix = load_close(VIX_TICKER)

r = (100 * np.log(spy).diff()).dropna().rename("r")        # daily log return, %
vix = vix.reindex(r.index).ffill()

print(f"{len(r):,} daily returns, {r.index[0].date()} to {r.index[-1].date()}")
r.plot(title=f"{TICKER} daily log returns (%)", lw=0.6);

## 2. Target and features
Everything in row $t$ is known at the close of day $t$, **except** the target, which looks 5 days ahead.

The features are built in logs because volatility is right-skewed and roughly log-normal, which tree models and regressions both handle better. The negative-return features capture the leverage effect. VIX is the market's own forward-looking volatility estimate, converted here to the same units as the target.

In [ ]:
r2 = r ** 2
df = pd.DataFrame({"r": r})

# Target: realized variance over t+1 .. t+HORIZON
df["y"] = r2.rolling(HORIZON).sum().shift(-HORIZON)

# HAR components (average daily variance over 1, 5, 22 days)
df["rv_d"] = r2
df["rv_w"] = r2.rolling(5).mean()
df["rv_m"] = r2.rolling(22).mean()

# Extra features for XGBoost
df["rv_q"] = r2.rolling(63).mean()
df["ret_5"] = r.rolling(5).sum()
df["ret_22"] = r.rolling(22).sum()
df["neg_rv_w"] = (r.clip(upper=0) ** 2).rolling(5).mean()      # downside variance
df["vix_var"] = vix ** 2 / 252 * HORIZON                        # VIX-implied 5-day variance, %^2

eps = 1e-4
for c in ["rv_d", "rv_w", "rv_m", "rv_q", "neg_rv_w", "vix_var"]:
    df[f"log_{c}"] = np.log(df[c] + eps)

XGB_FEATURES = ["log_rv_d", "log_rv_w", "log_rv_m", "log_rv_q", "log_neg_rv_w",
                "log_vix_var", "ret_5", "ret_22"]
HAR_FEATURES = ["rv_d", "rv_w", "rv_m"]

df = df.dropna(subset=HAR_FEATURES + XGB_FEATURES)
test_idx = df.loc[TEST_START:].dropna(subset=["y"]).index
print(f"Test period: {test_idx[0].date()} to {test_idx[-1].date()}  ({len(test_idx):,} forecasts)")

## 3. Walk-forward engine
**The lookahead trap:** on a refit date $d$, the targets for the last 5 days before $d$ aren't known yet, because they depend on returns after $d$. Training on them would leak the future. `walk_forward` drops ("purges") those rows. Forgetting this is one of the most common bugs in published-looking backtests.

In [ ]:
def walk_forward(fit_predict, features, refit_every):
    # Refit on an expanding window every `refit_every` days; forecast until the next refit.
    preds = pd.Series(index=test_idx, dtype=float)
    for start in range(0, len(test_idx), refit_every):
        block = test_idx[start:start + refit_every]
        refit_date = block[0]
        train = df.loc[:refit_date].iloc[:-1]           # strictly before the refit date
        train = train.iloc[:-HORIZON].dropna(subset=["y"])  # purge targets that overlap the future
        preds.loc[block] = fit_predict(train[features], train["y"], df.loc[block, features])
    return preds

forecasts = pd.DataFrame(index=test_idx)

## 4. Models
### 4a. Naive: recent average variance

In [ ]:
forecasts["Naive"] = df.loc[test_idx, "rv_m"] * HORIZON

### 4b. HAR-RV

In [ ]:
def har_fit_predict(X, y, X_new):
    model = sm.OLS(y, sm.add_constant(X)).fit()
    pred = model.predict(sm.add_constant(X_new, has_constant="add"))
    return np.maximum(pred, 1e-3)          # linear models can go negative; variance can't

forecasts["HAR"] = walk_forward(har_fit_predict, HAR_FEATURES, ML_REFIT_EVERY)

### 4c. GJR-GARCH(1,1), Student-t
`arch` handles walk-forward natively. `fit(last_obs=d)` estimates the parameters on data before `d`, and `forecast(start=d)` then produces a forecast at each later date using only the returns up to that date. Summing the 1- to 5-day-ahead variances gives the 5-day forecast.

In [ ]:
am = arch_model(r, mean="Constant", vol="GARCH", p=1, o=1, q=1, dist="t", rescale=False)
garch = pd.Series(index=test_idx, dtype=float)

for start in range(0, len(test_idx), GARCH_REFIT_EVERY):
    block = test_idx[start:start + GARCH_REFIT_EVERY]
    res = am.fit(last_obs=block[0], disp="off")
    fc = res.forecast(horizon=HORIZON, start=block[0], reindex=False)
    garch.loc[block] = fc.variance.loc[block].sum(axis=1)

forecasts["GARCH"] = garch
print(res.params.round(4))   # latest fitted parameters

### 4d. XGBoost
The model is trained on **log** variance and the forecast is mapped back with $\exp(\hat{y})$. That would systematically under-forecast (Jensen's inequality), so the prediction is scaled by the average of $\exp(\text{residual})$ on the training set (Duan's smearing correction).

The hyperparameters are deliberately conservative and **were not tuned on the test period**. Tuning them properly, with validation inside each training window, is one of the extensions at the end.

In [ ]:
def xgb_fit_predict(X, y, X_new):
    model = XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.03,
                         subsample=0.8, colsample_bytree=0.8, min_child_weight=20,
                         random_state=0, n_jobs=-1)
    log_y = np.log(y + 1e-4)
    model.fit(X, log_y)
    smear = np.mean(np.exp(log_y - model.predict(X)))
    xgb_fit_predict.last_model = model
    return np.exp(model.predict(X_new)) * smear

forecasts["XGBoost"] = walk_forward(xgb_fit_predict, XGB_FEATURES, ML_REFIT_EVERY)

## 5. Evaluation
- **MSE** is dominated by a few crisis weeks. It's useful to know, but it's noisy.
- **QLIKE** $= \frac{y}{f} - \ln\frac{y}{f} - 1$ penalizes under-forecasting more than over-forecasting, which suits risk management. Lower is better for both.
- **Mincer-Zarnowitz $R^2$:** how much of the variation in realized variance each forecast explains.
- **Diebold-Mariano:** is the gap in losses between two models statistically real? The overlapping 5-day targets make losses autocorrelated, so it uses Newey-West (HAC) errors.

In [ ]:
y = df.loc[test_idx, "y"].clip(lower=1e-6)

def qlike(y, f):
    return y / f - np.log(y / f) - 1

def mz_r2(y, f):
    return sm.OLS(y, sm.add_constant(f)).fit().rsquared

def dm_test(loss_a, loss_b):
    # Positive t-stat => model A has higher loss (B is better)
    d = (loss_a - loss_b).dropna()
    res = sm.OLS(d, np.ones(len(d))).fit(cov_type="HAC", cov_kwds={"maxlags": HORIZON})
    return res.tvalues.iloc[0], res.pvalues.iloc[0]

losses_q = forecasts.apply(lambda f: qlike(y, f))
losses_mse = forecasts.apply(lambda f: (y - f) ** 2)

summary = pd.DataFrame({
    "QLIKE": losses_q.mean(),
    "QLIKE vs Naive": losses_q.mean() / losses_q["Naive"].mean(),
    "MSE": losses_mse.mean(),
    "MSE vs Naive": losses_mse.mean() / losses_mse["Naive"].mean(),
    "MZ R2": forecasts.apply(lambda f: mz_r2(y, f)),
}).sort_values("QLIKE")
display(summary.round(3))

rows = []
for a in forecasts:
    for b in forecasts:
        if a < b:
            t, p = dm_test(losses_q[a], losses_q[b])
            better = b if t > 0 else a
            rows.append({"A": a, "B": b, "DM t-stat": round(t, 2), "p-value": round(p, 3),
                         "better (QLIKE)": better if p < 0.05 else "no significant difference"})
display(pd.DataFrame(rows))

## 6. What the forecasts look like
Shown as annualized volatility (%), which is easier to read than 5-day variance.

In [ ]:
to_ann_vol = lambda v: np.sqrt(v * 252 / HORIZON)

fig, ax = plt.subplots()
ax.plot(to_ann_vol(y), color="lightgray", lw=0.8, label="Realized (next 5 days)")
for col in ["HAR", "GARCH", "XGBoost"]:
    ax.plot(to_ann_vol(forecasts[col]), lw=1, label=col)
ax.set_ylabel("Annualized vol (%)"); ax.set_title("Forecast vs realized, full test period")
ax.legend(); plt.show()

window = slice("2020-01-15", "2020-06-30")
fig, ax = plt.subplots()
ax.plot(to_ann_vol(y.loc[window]), color="gray", lw=1.5, label="Realized")
for col in forecasts:
    ax.plot(to_ann_vol(forecasts.loc[window, col]), lw=1.2, label=col)
ax.set_title("Zoom: COVID crash, 2020"); ax.legend(); plt.show()

**When does XGBoost win or lose?** The chart below shows the cumulative QLIKE advantage of XGBoost over GARCH. A rising line means XGBoost is winning. Jumps usually line up with regime changes, which is where models differ most.

In [ ]:
cum = (losses_q["GARCH"] - losses_q["XGBoost"]).cumsum()
ax = cum.plot(title="Cumulative QLIKE advantage of XGBoost over GARCH (up = XGBoost better)")
ax.axhline(0, color="k", lw=0.5); plt.show()

importance = pd.Series(xgb_fit_predict.last_model.feature_importances_, index=XGB_FEATURES)
importance.sort_values().plot.barh(title="XGBoost feature importance (latest refit)"); plt.show()

## 7. Findings
*Write your interpretation here once you've run the notebook. Questions to answer:*
- Did any model significantly beat HAR? (Beating only Naive doesn't count.)
- How much of XGBoost's edge, if any, comes from the VIX? Rerun without `log_vix_var` to find out.
- Which model reacted fastest in March 2020, and which overshot on the way down?

## 8. Extensions (turning this into a month-plus project)
1. **Better realized variance.** Daily squared returns are a very noisy measure of the truth. Intraday 5-minute realized variance would sharpen every result.
2. **Proper tuning.** Use time-series cross-validation *inside* each training window for the XGBoost hyperparameters.
3. **More assets:** QQQ, IWM, TLT, GLD, sector ETFs. Is the ranking of the models stable?
4. **Ablations:** drop one feature group at a time to measure what each contributes.
5. **More features:** VIX term structure (`^VIX3M`), credit spreads and yields from FRED, calendar effects such as FOMC days.
6. **Economic value:** use each forecast to run a volatility-targeting strategy, then compare Sharpe ratios and drawdowns after costs. That's the question a portfolio manager actually cares about.